# Day 2: Master Database Loading
This notebook loads all cleaned datasets from `data/processed/` into the SQLite database `bluestock_mf.db` using the star schema.

In [ ]:
import pandas as pd
import sqlite3
import os
from sqlalchemy import create_engine, text

# Paths
processed_dir = r'../../data/day2_processed'
db_path = r'../../bluestock_mf.db'
schema_path = r'../../sql/schema.sql'

def load_master():
    # 1. Initialize Database
    print("Initializing database...")
    if os.path.exists(db_path):
        try:
            os.remove(db_path)
        except:
            pass
    
    conn = sqlite3.connect(db_path)
    with open(schema_path, 'r') as f:
        schema_sql = f.read()
    conn.executescript(schema_sql)
    conn.close()
    
    engine = create_engine(f'sqlite:///{db_path}')
    
    # 2. Load Dimensions
    print("Loading Dimensions...")
    df_perf = pd.read_csv(os.path.join(processed_dir, 'day2_07_scheme_performance_cleaning.csv'))
    dim_fund = df_perf[['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan']].drop_duplicates()
    dim_fund.to_sql('dim_fund', engine, if_exists='append', index=False)
    
    # 3. Load Fact Tables
    print("Loading Fact Tables...")
    
    # fact_nav
    df_nav = pd.read_csv(os.path.join(processed_dir, 'day2_02_nav_history_cleaning.csv'))
    df_nav.to_sql('fact_nav', engine, if_exists='append', index=False)
    
    # fact_transactions
    df_trans = pd.read_csv(os.path.join(processed_dir, 'day2_08_investor_transactions_cleaning.csv'))
    df_trans.to_sql('fact_transactions', engine, if_exists='append', index=False)
    
    # fact_performance
    fact_perf_cols = [
        'amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 
        'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 
        'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 
        'morningstar_rating', 'risk_grade', 'expense_ratio_anomaly'
    ]
    df_perf[fact_perf_cols].to_sql('fact_performance', engine, if_exists='append', index=False)

    # fact_aum
    df_aum = pd.read_csv(os.path.join(processed_dir, 'day2_03_aum_by_fund_house_cleaning.csv'))
    df_aum[['fund_house', 'aum_crore', 'date']].to_sql('fact_aum', engine, if_exists='append', index=False)
    
    # Load Additional Tables for 100% Compliance
    print("Loading Additional Tables...")
    
    # fact_category_inflows
    df_cat = pd.read_csv(os.path.join(processed_dir, 'day2_05_category_inflows_cleaning.csv'))
    df_cat.to_sql('fact_category_inflows', engine, if_exists='append', index=False)
    
    # fact_folio_count
    df_folio = pd.read_csv(os.path.join(processed_dir, 'day2_06_industry_folio_count_cleaning.csv'))
    df_folio.to_sql('fact_folio_count', engine, if_exists='append', index=False)
    
    # fact_portfolio_holdings
    df_holdings = pd.read_csv(os.path.join(processed_dir, 'day2_09_portfolio_holdings_cleaning.csv'))
    df_holdings.to_sql('fact_portfolio_holdings', engine, if_exists='append', index=False)
    
    # fact_benchmark_indices
    df_bench = pd.read_csv(os.path.join(processed_dir, 'day2_10_benchmark_indices_cleaning.csv'))
    df_bench.to_sql('fact_benchmark_indices', engine, if_exists='append', index=False)
    
    # 4. Load dim_date
    print("Loading dim_date...")
    all_dates = pd.concat([df_nav['date'], df_trans['transaction_date']]).unique()
    dim_date = pd.DataFrame({'date': all_dates})
    dim_date['date'] = pd.to_datetime(dim_date['date'])
    dim_date['day'] = dim_date['date'].dt.day
    dim_date['month'] = dim_date['date'].dt.month
    dim_date['year'] = dim_date['date'].dt.year
    dim_date['quarter'] = dim_date['date'].dt.quarter
    dim_date['day_of_week'] = dim_date['date'].dt.day_name()
    dim_date['date'] = dim_date['date'].dt.strftime('%Y-%m-%d')
    dim_date.to_sql('dim_date', engine, if_exists='append', index=False)
    
    print("\nLoading Complete!")
    
    # Verification
    print("\nVerification (Row Counts):")
    with engine.connect() as conn:
        for table in ['dim_fund', 'dim_date', 'fact_nav', 'fact_transactions', 'fact_performance', 'fact_aum']:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
            print(f"{table}: {count} rows")
            
load_master()